# dfs-three-set-toposort — ex2: implement iterative (stack-based) three-set DFS toposort

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dfs-three-set-toposort`. Running the final beacon cell reports progress against the `Backprop: DFS three-set toposort` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: DFS three-set toposort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dfs-three-set-toposort`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dfs-three-set-toposort"
DD_SUBTOPIC = "Backprop: DFS three-set toposort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DFS three-set toposort — iterative form

The recursive three-color DFS works great until you hit Python's default recursion limit (1000) on a deep computational graph. The iterative form uses an explicit stack and two-phase node entries:

```python
def topological_sort(root, get_children):
    result = []
    perm = set()
    temp = set()
    # Stack frames: (node, 'enter') or (node, 'exit').
    stack = [(root, 'enter')]
    while stack:
        node, phase = stack.pop()
        nid = id(node)
        if phase == 'exit':
            temp.discard(nid); perm.add(nid); result.append(node)
            continue
        if nid in perm: continue
        if nid in temp: raise ValueError('cycle')
        temp.add(nid)
        stack.append((node, 'exit'))     # post-order hook
        for child in get_children(node):
            stack.append((child, 'enter'))
    return result
```

Same deps-first output as the recursive version (root LAST). The two-phase trick — push an `exit` marker BEFORE the children — gives us the post-order moment for `result.append` without recursion.

### Exercise 2 — implement iterative (stack-based) three-set DFS toposort

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the iterative (stack-based) variant of three-color DFS topological sort using two-phase node entries so deep computational graphs don't hit Python's recursion limit.
> Keywords: dfs, topological-sort, iterative, explicit-stack, deep-graph
> ```

**KCs targeted:** `dfs-three-set-toposort`, `cycle-detection-temp-set`

Implement `topological_sort_iter(root, get_children)` — same contract as the recursive version from ex1, but with an EXPLICIT stack so it doesn't recurse into Python.

**Why this matters.** A real-world neural-network forward graph can be hundreds of layers deep. Python's default `sys.setrecursionlimit` is 1000 frames — the recursive form throws `RecursionError` on a 1024-layer DAG. Production autograd libraries (including PyTorch's C++ engine) use iterative traversal for exactly this reason.

**Contract.** Same as the recursive version:
- Returns descendants of `root` in deps-FIRST order (root LAST).
- Each reachable node appears EXACTLY once (diamond DAGs OK).
- Raises `ValueError` on a cycle (DAG only).
- Key by `id(node)`.

**Two-phase stack trick.** Pre-order tells you which node is ENTERING the DFS frame; post-order tells you which node has FINISHED its subtree. Recursive code gets both for free. Iterative: push an `('exit', node)` marker BEFORE pushing the children, so when we pop the `'exit'` we know the subtree is done.

```python
stack = [(root, 'enter')]
while stack:
    node, phase = stack.pop()
    if phase == 'exit':       # subtree done — post-order moment
        temp.discard(id(node)); perm.add(id(node)); result.append(node)
        continue
    if id(node) in perm: continue
    if id(node) in temp: raise ValueError('cycle')
    temp.add(id(node))
    stack.append((node, 'exit'))           # post-order hook
    for child in get_children(node):
        stack.append((child, 'enter'))
```

DO NOT use `sys.setrecursionlimit` — that's the wrong fix. The test explicitly builds a graph deeper than `sys.getrecursionlimit()`.

In [ ]:
def topological_sort_iter(root, get_children):
    """Iterative DFS topo sort. Same contract as the recursive version:
    descendants of root in deps-first order (root LAST), unique nodes,
    cycle -> ValueError.
    """
    raise NotImplementedError()


def _test_ex2():
    import sys as _sys

    # --- helper node ---
    class N:
        def __init__(self, name, *children):
            self.name = name
            self.children = list(children)
        def __repr__(self):
            return f'N({self.name})'

    def get_children(n):
        return n.children

    # --- linear chain a -> b -> c ---
    c = N('c')
    b = N('b', c)
    a = N('a', b)
    order = topological_sort_iter(a, get_children)
    names = [n.name for n in order]
    assert names == ['c', 'b', 'a'], f'linear chain order: {names}'

    # --- diamond DAG ---
    d = N('d')
    b = N('b', d)
    c = N('c', d)
    a = N('a', b, c)
    order = topological_sort_iter(a, get_children)
    names = [n.name for n in order]
    assert names.count('d') == 1, f'd must appear ONCE: {names}'
    assert names[-1] == 'a'
    assert names.index('d') < names.index('b')
    assert names.index('d') < names.index('c')
    assert len(order) == 4

    # --- THE iterative-form payoff: deep chain that would blow Python
    #     recursion (default ~1000). Build a chain ~2000 deep and assert
    #     the iterative form completes without RecursionError. ---
    depth = max(2000, _sys.getrecursionlimit() + 500)
    deep_leaf = N('leaf')
    cur = deep_leaf
    for i in range(depth):
        cur = N(f'n{i}', cur)
    deep_root = cur
    order = topological_sort_iter(deep_root, get_children)
    assert len(order) == depth + 1, (
        f'deep chain should yield depth+1 nodes, got {len(order)}'
    )
    assert order[0] is deep_leaf, 'leaf must be FIRST (deps-first)'
    assert order[-1] is deep_root, 'root must be LAST'

    # --- cycle detection still works ---
    x = N('x')
    y = N('y')
    x.children = [y]
    y.children = [x]
    raised = False
    try:
        topological_sort_iter(x, get_children)
    except ValueError:
        raised = True
    assert raised, 'cycle must still raise ValueError in iterative form'

    # --- self-loop ---
    s = N('s')
    s.children = [s]
    raised = False
    try:
        topological_sort_iter(s, get_children)
    except ValueError:
        raised = True
    assert raised, 'self-loop must raise ValueError'

    # --- singleton ---
    lonely = N('lonely')
    order = topological_sort_iter(lonely, get_children)
    assert order == [lonely]

    # --- branching: ensure deps-first invariant holds ---
    leaf = N('leaf')
    m1 = N('m1', leaf)
    m2 = N('m2', leaf)
    root = N('root', m1, m2)
    order = topological_sort_iter(root, get_children)
    names = [n.name for n in order]
    pos = {nm: i for i, nm in enumerate(names)}
    assert pos['leaf'] < pos['m1'] < pos['root']
    assert pos['leaf'] < pos['m2'] < pos['root']
    assert names[-1] == 'root'
    assert len(order) == 4   # leaf, m1, m2, root — each once

    # --- still uses id() (not value-eq) — value-equal nodes are distinct ---
    class Vn:
        def __init__(self, name, *children):
            self.name = name; self.children = list(children)
        def __eq__(self, other): return self.name == getattr(other, 'name', None)
        def __hash__(self): return hash(self.name)
    x1 = Vn('x'); x2 = Vn('x')   # value-equal but DIFFERENT objects
    rt = Vn('rt', x1, x2)
    order = topological_sort_iter(rt, lambda n: n.children)
    # id() keying should keep them DISTINCT — three nodes, not two.
    assert len(order) == 3, (
        f'id()-keying must keep value-equal nodes distinct: {[n.name for n in order]}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def topological_sort_iter(root, get_children):
    result = []
    perm = set()
    temp = set()
    # Stack frames: (node, 'enter') or (node, 'exit').
    stack = [(root, 'enter')]
    while stack:
        node, phase = stack.pop()
        nid = id(node)
        if phase == 'exit':
            # subtree finished — drop from temp, mark perm, emit
            temp.discard(nid)
            if nid not in perm:
                perm.add(nid)
                result.append(node)
            continue
        # phase == 'enter'
        if nid in perm:
            continue
        if nid in temp:
            raise ValueError(
                f'Cycle detected at {node!r} — graph is not a DAG'
            )
        temp.add(nid)
        # push our own exit marker BEFORE pushing children, so we
        # finalize after all children finalize (LIFO stack order).
        stack.append((node, 'exit'))
        for child in get_children(node):
            stack.append((child, 'enter'))
    return result
```

**Why two phases.** A recursive DFS gets two natural moments: the call (pre-order) and the return (post-order). Iterative code only has 'I popped this off the stack' — so we encode the two moments as two stack entries. The `'exit'` marker pushed BEFORE the children ensures it's popped AFTER they all finish (LIFO).

**Why `temp.discard(nid)` in `'exit'`.** If we just `temp.remove`, and a node was re-popped after already being in `perm` (rare edge case if children are pushed multiple times), `remove` would raise. `discard` is the no-op-on-absent variant — safer.

**Why the `if nid not in perm` guard on emit.** With diamond DAGs, the same child can be queued from multiple parents. The `'enter'`-phase guard `if nid in perm: continue` rejects the re-entry, so the `'exit'` is never pushed a second time — but the guard on `'exit'` is a cheap belt-and-braces for any unusual `get_children` callback.

**Sibling visitation order.** Because we push children left-to-right and pop right-to-left, the iterative version visits siblings in REVERSE order vs the recursive form. The deps-first contract is unchanged; only the order of ties between independent leaves differs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()